# Chicago taxi trips — week of 11–17 September 2023

**Summary (filled in at the end of the analysis)**

SUMMARY_PLACEHOLDER

## 1. Data loading

All figures describe **trips reported to the City of Chicago**, not total market volume (per `dataset.md`, not every trip is reported).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('figures/analysis', exist_ok=True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Chart style: one blue for the main series, orange only for a deliberate highlight, recessive grid
BLUE, ORANGE, GREY, INK = '#2a78d6', '#eb6834', '#b9b8b3', '#0b0b0b'
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': '#e6e5e1', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'figure.dpi': 100, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
})

raw = pd.read_csv(
    'trips.csv',
    dtype={'pickup_census_tract': 'string', 'dropoff_census_tract': 'string'},
    parse_dates=['trip_start_timestamp', 'trip_end_timestamp'],
)
print('shape:', raw.shape)
print('trip_id unique:', raw.trip_id.is_unique)
print('start range:', raw.trip_start_timestamp.min(), '->', raw.trip_start_timestamp.max())

shape: (135768, 23)
trip_id unique: True
start range: 2023-09-11 00:00:00 -> 2023-09-17 23:45:00


## 2. Data checks and cleaning

### 2.1 Missing values

In [2]:
miss = raw.isna().sum()
miss = miss[miss > 0].to_frame('missing')
miss['pct_of_rows'] = (miss.missing / len(raw) * 100).round(2)
miss

,missing,pct_of_rows
trip_end_timestamp,1,0.00
trip_seconds,33,0.02
pickup_census_tract,71404,52.59
dropoff_census_tract,73842,54.39
pickup_community_area,3815,2.81
dropoff_community_area,13794,10.16
fare,324,0.24
tips,324,0.24
tolls,324,0.24
extras,324,0.24


Notes: the five money columns (`fare`, `tips`, `tolls`, `extras`, `trip_total`) are all missing together on the same rows. The census-tract and dropoff columns are not used in this analysis (the brief says to use community areas).

### 2.2 Implausible or inconsistent values

In [3]:
money = ['fare', 'tips', 'tolls', 'extras', 'trip_total']
has_money = raw.trip_total.notna()
parts_gap = (raw.trip_total - raw[['fare', 'tips', 'tolls', 'extras']].sum(axis=1)).round(2)
speed_mph = raw.trip_miles / (raw.trip_seconds / 3600)

checks = pd.DataFrame([
    ('Money fields missing (all five together)',          (~has_money).sum()),
    ('fare = 0 or trip_total = 0 (money present)',        (has_money & ((raw.fare <= 0) | (raw.trip_total <= 0))).sum()),
    ('trip_total > $500',                                 (raw.trip_total > 500).sum()),
    ('trip_total > sum of fare+tips+tolls+extras',        (parts_gap > 0.01).sum()),
    ('trip_total < sum of parts',                         (parts_gap < -0.01).sum()),
    ('   ... largest gap above the sum of parts ($)',     parts_gap.max()),
    ('trip_miles = 0',                                    (raw.trip_miles == 0).sum()),
    ('trip_miles > 100',                                  (raw.trip_miles > 100).sum()),
    ('trip_seconds = 0',                                  (raw.trip_seconds == 0).sum()),
    ('trip_seconds missing',                              raw.trip_seconds.isna().sum()),
    ('trip_seconds > 6 hours',                            (raw.trip_seconds > 6 * 3600).sum()),
    ('Implied speed > 100 mph (seconds > 0)',             (speed_mph > 100).sum()),
    ('extras > $100',                                     (raw.extras > 100).sum()),
    ('Duplicate rows apart from trip_id',                 raw.duplicated([c for c in raw.columns if c != 'trip_id']).sum()),
], columns=['check', 'rows']).set_index('check')
checks['rows'] = [f'${v:,.2f}' if 'largest gap' in i else f'{int(v):,}' for i, v in checks['rows'].items()]
checks

,rows
check,
Money fields missing (all five together),324
fare = 0 or trip_total = 0 (money present),135
trip_total > $500,8
trip_total > sum of fare+tips+tolls+extras,"48,197"
trip_total < sum of parts,0
... largest gap above the sum of parts ($),$3.00
trip_miles = 0,"13,993"
trip_miles > 100,11
trip_seconds = 0,"2,731"


### 2.3 Cleaning decisions and assumptions

Principle: **exclude a trip only if its money fields cannot be trusted** (every answer except the pure trip counts depends on them), and apply the *same* cleaned table to all four questions so numbers agree across sections. Odd distance/duration values are flagged but **not** excluded, because no question uses `trip_miles` or `trip_seconds`.

| # | Issue | Treatment |
|---|---|---|
| A1 | All five money fields missing (mostly cash trips) | **Exclude** — no revenue can be computed |
| A2 | `fare` = 0 or `trip_total` = 0 | **Exclude** — nothing billed; mostly cash flag-drop records |
| A3 | `trip_total` > $500 (>20× the median) | **Exclude** — dominated by obvious entry errors (e.g. $4,444 tolls on a 1-mile trip) |
| A4 | `trip_total` is above the sum of its parts (always by ≤ $3, never below) | **Keep**, use `trip_total` as reported. The gap looks like an un-itemised surcharge (usually exactly $0.50), not an error |
| A5 | Zero or extreme `trip_miles` / `trip_seconds`, implied speed > 100 mph | **Keep** (flag only). Fares on these rows look normal; the distance/time fields are unreliable but unused |
| A6 | 92 duplicate rows apart from `trip_id` | **Keep**. Almost all are $3.25, 0-mile, 0-second cash records (repeated meter starts); `trip_id`s are unique, so we cannot tell them from genuine separate trips |
| A7 | `extras` > $100 (15 rows) | **Keep**; immaterial to totals, noted as a data-quality oddity |
| A8 | Company names with spelling variants | **Merge** the two unambiguous pairs (see below). "Taxi Affiliation Services Llc - Yell…" is left separate because it is truncated and could be a distinct brand |
| A9 | Pickup community area blank | Kept as "no pickup community area" (see Q2) — not imputed |

Other assumptions: timestamps are Chicago local time; `Prcard` is read as a prepaid card; `Mobile` as app-based payment; `Unknown`, `No Charge` and `Dispute` are kept as reported categories.

In [4]:
n0 = len(raw)
a1 = raw.trip_total.isna()
a2 = ~a1 & ((raw.fare <= 0) | (raw.trip_total <= 0))
a3 = ~a1 & ~a2 & (raw.trip_total > 500)
steps = [('Raw trips', 0), ('A1 money fields missing', a1.sum()), ('A2 zero fare or zero total', a2.sum()), ('A3 trip_total > $500', a3.sum())]

df = raw[~(a1 | a2 | a3)].copy()

company_map = {
    'Taxicab Insurance Agency, LLC': 'Taxicab Insurance Agency Llc',
    'Choice Taxi Association Inc': 'Choice Taxi Association',
}
df['company'] = df.company.replace(company_map)

df['start_date'] = df.trip_start_timestamp.dt.normalize()
df['dow'] = df.trip_start_timestamp.dt.day_name()
df['hour'] = df.trip_start_timestamp.dt.hour
df['weekend'] = df.trip_start_timestamp.dt.dayofweek >= 5

tbl = pd.DataFrame(steps, columns=['step', 'rows_removed'])
tbl['rows_remaining'] = n0 - tbl.rows_removed.cumsum()
tbl['rows_removed'] = tbl.rows_removed.map(lambda v: f'{v:,}' if v else '')
tbl['rows_remaining'] = tbl.rows_remaining.map('{:,}'.format)
print(tbl.to_string(index=False))
print(f'\nExcluded in total: {n0 - len(df):,} rows ({(n0 - len(df)) / n0:.2%}); cleaned table: {len(df):,} trips')
print('Company names merged (A8):', {k: int((raw.company == k).sum()) for k in company_map}, '-> now', df.company.nunique(), 'distinct companies (was', raw.company.nunique(), ')')
print('Taxi_ids appearing under more than one merged company:', int((df.groupby('taxi_id').company.nunique() > 1).sum()))

                      step rows_removed rows_remaining
                 Raw trips                     135,768
   A1 money fields missing          324        135,444
A2 zero fare or zero total          135        135,309
      A3 trip_total > $500            8        135,301

Excluded in total: 467 rows (0.34%); cleaned table: 135,301 trips
Company names merged (A8): {'Taxicab Insurance Agency, LLC': 2379, 'Choice Taxi Association Inc': 1064} -> now 30 distinct companies (was 32 )
Taxi_ids appearing under more than one merged company: 3


/tmp/ipykernel_289/4150180879.py:18: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['weekend'] = df.trip_start_timestamp.dt.dayofweek >= 5
